# Exercise 4 — safe_ask

`safe_ask` chains all four guardrails: validate the input, check the budget, seek approval, run the agent, validate the output.  Any failed layer returns a blocked record with a reason; nothing raises.  The `agent_fn(query, llm_fn=None) -> str` signature makes the real LLM call swappable for testing.

In [ ]:
import json
def validate_text(text, max_length=None, banned=None):
    text_str = str(text)
    if max_length is not None and len(text_str) > max_length:
        return False, ("text exceeds max_length ("
                       + str(len(text_str)) + " > " + str(max_length) + " chars)")
    if banned:
        lower = text_str.lower()
        for pattern in banned:
            if str(pattern).lower() in lower:
                return False, "banned pattern found: " + repr(pattern)
    return True, ""

class Guard:
    def __init__(self, max_length=None, banned=None):
        self.max_length = max_length
        self.banned = list(banned) if banned else []
    def check(self, text):
        return validate_text(text, self.max_length, self.banned)
class ApprovalGate:
    def __init__(self, approve_fn=None):
        self._approve_fn = approve_fn if approve_fn is not None else (lambda action: True)
    def check(self, action):
        try:
            result = bool(self._approve_fn(str(action)))
        except Exception:
            result = False
        return (True, "approved") if result else (False, "rejected by approval gate")
class BudgetTracker:
    def __init__(self, max_calls=None):
        self.max_calls = max_calls
        self._count = 0
    def ok(self):
        if self.max_calls is not None and self._count >= self.max_calls:
            return False, ("budget exceeded (" + str(self._count)
                           + "/" + str(self.max_calls) + " calls)")
        return True, ""
    def record(self): self._count += 1
    def reset(self): self._count = 0
    @property
    def count(self): return self._count
_echo_agent = lambda query, llm_fn=None: 'Answer: ' + str(query)

# ── Exercise: implement safe_ask ─────────────────────────────────────────────

def safe_ask(query, agent_fn, input_guard=None, output_guard=None,
             budget=None, gate=None, llm_fn=None):
    # TODO: pipeline:
    # 1. record = {"query": query, "answer": None, "blocked": False, "reason": ""}
    # 2. input_guard: if not None, check query; if fails, set blocked+reason, return
    # 3. budget: if not None, check ok(); if fails, set blocked+reason, return;
    #            if passes, call budget.record()
    # 4. gate: if not None, check query; if not approved, set blocked+reason, return
    # 5. try: answer = str(agent_fn(query, llm_fn=llm_fn))
    #    except: answer = "Error: " + str(exc)
    # 6. output_guard: if not None, check answer; if fails, set blocked+reason,
    #                  record["answer"] = "[blocked]", return
    # 7. record["answer"] = answer; return record
    return {"query": query, "answer": None, "blocked": False, "reason": ""}


### Checks

In [ ]:
checks = 0

# 1 — not blocked when all guards pass
try:
    r = safe_ask("hello", _echo_agent)
    assert not r["blocked"] and r["answer"] == "Answer: hello"
    checks += 1; print("✅ 1 safe_ask passes through with no guards")
except Exception as e:
    print("❌ 1:", e)

# 2 — input guard blocks long query
try:
    guard = Guard(max_length=5)
    r = safe_ask("this is too long", _echo_agent, input_guard=guard)
    assert r["blocked"] and "input" in r["reason"]
    assert r["answer"] is None
    checks += 1; print("✅ 2 input guard blocks long query")
except Exception as e:
    print("❌ 2:", e)

# 3 — budget blocks after max_calls
try:
    b = BudgetTracker(max_calls=2)
    safe_ask("q1", _echo_agent, budget=b)
    safe_ask("q2", _echo_agent, budget=b)
    r3 = safe_ask("q3", _echo_agent, budget=b)
    assert r3["blocked"] and "budget" in r3["reason"]
    checks += 1; print("✅ 3 budget blocks after max_calls")
except Exception as e:
    print("❌ 3:", e)

# 4 — gate blocks when approve_fn returns False
try:
    gate = ApprovalGate(approve_fn=lambda a: False)
    r = safe_ask("dangerous query", _echo_agent, gate=gate)
    assert r["blocked"] and "gate" in r["reason"]
    checks += 1; print("✅ 4 gate blocks when approve_fn returns False")
except Exception as e:
    print("❌ 4:", e)

# 5 — output guard blocks and sets answer to "[blocked]"
try:
    out_guard = Guard(banned=["secret"])
    leaky_agent = lambda q, llm_fn=None: "Your password secret is 1234"
    r = safe_ask("tell me something", leaky_agent, output_guard=out_guard)
    assert r["blocked"] and "output" in r["reason"]
    assert r["answer"] == "[blocked]"
    checks += 1; print("✅ 5 output guard blocks and replaces answer with '[blocked]'")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
